# YOLO Character Detection — Training Pipeline

This notebook implements a **two-stage curriculum training** strategy for a YOLOv8-Small character detector:

1. **Stage 1 — Synthetic Pre-training:** Train on 10,000 programmatically rendered multi-script page images (generated by `kaggle_text_classifier_data_generation.py`). This gives the model broad character-localisation priors.
2. **Stage 2 — Real-data Fine-tuning:** Fine-tune the Stage 1 checkpoint on the real competition annotated images using a very low learning rate to preserve learned representations.

**Output of this notebook:** `intermediate_boxes_expanded.json` — used as input to `text-classification.ipynb`.

> **Data source:** The synthetic dataset is saved to `dataset_production_test1` by the data generation script. On Kaggle it is mounted as a notebook output dataset. See Step 3 for path configuration details.

## Step 1 — Install Dependencies

Install `ultralytics` (YOLOv8). All other required libraries (`torch`, `torchvision`, `Pillow`, `PyYAML`, `numpy`, `opencv-python`) are pre-installed in the Kaggle environment.

In [1]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 100.2 MB/s eta 0:00:0000:010:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numb

## Step 2 — Hardware Probe

Verify that the Kaggle GPU accelerator is enabled and detect how many GPUs are available. Stage 1 training uses Distributed Data Parallel (DDP) across both T4 GPUs when available. If only one GPU is detected, Stage 1 will fall back to single-GPU mode automatically.

In [2]:
import os
import torch
import subprocess

print("--- HARDWARE PROBE ---")

# 1. Ask Nvidia directly what hardware is physically attached
try:
    nvidia_smi = subprocess.check_output("nvidia-smi -L", shell=True).decode('utf-8')
    print("NVIDIA-SMI Detects:")
    print(nvidia_smi.strip())
except Exception as e:
    print(f"Failed to query nvidia-smi: {e}")

print("\n--- PYTORCH ENVIRONMENT ---")

# 2. Check what Kaggle told the environment it is allowed to see
visible_devices = os.getenv('CUDA_VISIBLE_DEVICES', 'Not Set')
print(f"CUDA_VISIBLE_DEVICES Environment Variable: {visible_devices}")

# 3. Check what PyTorch actually sees
num_gpus = torch.cuda.device_count()
print(f"PyTorch Detects {num_gpus} GPU(s).")

for i in range(num_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
    
if num_gpus < 2:
    print("\n[WARNING] Kaggle has not provided two GPUs. Training will be forced to single GPU.")
else:
    print("\n[SUCCESS] Both GPUs are active and ready for Distributed Data Parallel (DDP) training!")

--- HARDWARE PROBE ---
NVIDIA-SMI Detects:
GPU 0: Tesla T4 (UUID: GPU-b32009a4-7012-a894-8ae0-5fb9907cb679)
GPU 1: Tesla T4 (UUID: GPU-9b04fe6f-3632-7b62-2666-2e06cd85dae9)

--- PYTORCH ENVIRONMENT ---
CUDA_VISIBLE_DEVICES Environment Variable: Not Set
PyTorch Detects 2 GPU(s).
  GPU 0: Tesla T4
  GPU 1: Tesla T4

[SUCCESS] Both GPUs are active and ready for Distributed Data Parallel (DDP) training!


## Step 3 — Configuration

Set the two key path variables for this notebook:

| Variable | Description |
| :--- | :--- |
| `INPUT_DIR` | Path to the `dataset_production_test1` directory produced by the synthetic data generation script. On Kaggle this is a notebook output dataset mounted under `/kaggle/input/`. |
| `WORKING_DIR` | Writable directory where the YOLO-compatible folder structure will be assembled. On Kaggle this is `/kaggle/working/Custom_Dataset`. |

> **Path adjustment:** If you are running this notebook locally (not on Kaggle), update both variables to match your local filesystem paths before proceeding.

In [3]:
import os
import shutil
import random
import yaml
from ultralytics import YOLO

# --- 1. CONFIGURATION ---
# The path where your generated data currently lives
INPUT_DIR = "/kaggle/input/notebooks/thenamelessmonster/comsys7-datagen/dataset_production_test1"

# The target YOLO directory structure we will build in Kaggle's writable space
WORKING_DIR = "/kaggle/working/Custom_Dataset"

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


### Dataset Preparation

**Note:** The synthetic dataset generated by `kaggle_text_classifier_data_generation.py` is saved to `dataset_production_test1`.
Because the script was run remotely, its output was uploaded as an input dataset to this notebook.
The code below copies it to `/kaggle/working/Custom_Dataset`.

> **Important:** If you are running this locally (not on Kaggle), or if the dataset has not been copied to the working directory yet, you MUST uncomment the code block below. Please adjust the `INPUT_DIR` and `WORKING_DIR` paths accordingly before running it.

In [4]:
"""
IMAGES_TRAIN = os.path.join(WORKING_DIR, "images", "train")
IMAGES_VAL = os.path.join(WORKING_DIR, "images", "val")
LABELS_TRAIN = os.path.join(WORKING_DIR, "labels", "train")
LABELS_VAL = os.path.join(WORKING_DIR, "labels", "val")

# Split ratio: 80% for training, 20% for validation
SPLIT_RATIO = 0.8  

# --- 2. BUILD FOLDER STRUCTURE ---
print("Building YOLO directory structure...")
for path in [IMAGES_TRAIN, IMAGES_VAL, LABELS_TRAIN, LABELS_VAL]:
    os.makedirs(path, exist_ok=True)

# --- 3. SHUFFLE AND SPLIT DATA ---
print(f"Scanning for files in: {INPUT_DIR}")
# Get all image files (assuming they are .jpg)
all_images = [f for f in os.listdir(INPUT_DIR) if f.endswith('.jpg')]
random.shuffle(all_images) # Crucial to randomize fonts and languages!

split_index = int(len(all_images) * SPLIT_RATIO)
train_images = all_images[:split_index]
val_images = all_images[split_index:]

print(f"Total pairs found: {len(all_images)}")
print(f"Allocating {len(train_images)} to Train, {len(val_images)} to Validation.")

# --- 4. COPY FILES (Images + Matching Text Files) ---
def copy_files(file_list, target_img_dir, target_label_dir):
    for img_name in file_list:
        # Define paths for the image
        src_img = os.path.join(INPUT_DIR, img_name)
        dst_img = os.path.join(target_img_dir, img_name)
        
        # Define paths for the matching YOLO label file
        txt_name = img_name.replace('.jpg', '.txt')
        src_txt = os.path.join(INPUT_DIR, txt_name)
        dst_txt = os.path.join(target_label_dir, txt_name)
        
        # Copy the files if both exist
        if os.path.exists(src_img) and os.path.exists(src_txt):
            shutil.copy(src_img, dst_img)
            shutil.copy(src_txt, dst_txt)
        else:
            print(f"Warning: Missing pair for {img_name}")

print("\nCopying Training files...")
copy_files(train_images, IMAGES_TRAIN, LABELS_TRAIN)
print("Copying Validation files...")
copy_files(val_images, IMAGES_VAL, LABELS_VAL)
"""

'\nIMAGES_TRAIN = os.path.join(WORKING_DIR, "images", "train")\nIMAGES_VAL = os.path.join(WORKING_DIR, "images", "val")\nLABELS_TRAIN = os.path.join(WORKING_DIR, "labels", "train")\nLABELS_VAL = os.path.join(WORKING_DIR, "labels", "val")\n\n# Split ratio: 80% for training, 20% for validation\nSPLIT_RATIO = 0.8  \n\n# --- 2. BUILD FOLDER STRUCTURE ---\nprint("Building YOLO directory structure...")\nfor path in [IMAGES_TRAIN, IMAGES_VAL, LABELS_TRAIN, LABELS_VAL]:\n    os.makedirs(path, exist_ok=True)\n\n# --- 3. SHUFFLE AND SPLIT DATA ---\nprint(f"Scanning for files in: {INPUT_DIR}")\n# Get all image files (assuming they are .jpg)\nall_images = [f for f in os.listdir(INPUT_DIR) if f.endswith(\'.jpg\')]\nrandom.shuffle(all_images) # Crucial to randomize fonts and languages!\n\nsplit_index = int(len(all_images) * SPLIT_RATIO)\ntrain_images = all_images[:split_index]\nval_images = all_images[split_index:]\n\nprint(f"Total pairs found: {len(all_images)}")\nprint(f"Allocating {len(train_imag

## Step 5 — Generate YOLO Dataset YAML

Create the `data.yaml` configuration file that tells Ultralytics where to find the training and validation images. This file uses a **single class** (`Character`, class index 0) because the detector only needs to localise characters — classification is handled by a separate model downstream.

In [5]:
# --- 5. GENERATE THE YOLO YAML MAP ---
yaml_path = os.path.join(WORKING_DIR, "data.yaml")
yaml_content = {
    'path': WORKING_DIR,
    'train': 'images/train',
    'val': 'images/val',
    'names': {0: 'Character'}
}

with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print(f"\n[SUCCESS] data.yaml generated at {yaml_path}")
print("Data Preparation Complete! Initiating YOLO Training...")


[SUCCESS] data.yaml generated at /kaggle/working/Custom_Dataset/data.yaml
Data Preparation Complete! Initiating YOLO Training...


## Step 6 — Stage 1: Synthetic Data Pre-training

Train YOLOv8-Small on the 10,000-image synthetic dataset. Key hyperparameters:

| Parameter | Value | Rationale |
| :--- | :--- | :--- |
| `epochs` | 10 | Short run to build general character-detection features without fitting to synthetic rendering artefacts |
| `imgsz` | 1024 | High resolution to preserve small character detail in A4-scale page images |
| `batch` | 8 | Balanced for dual-T4 Kaggle GPU configuration |
| `device` | `[0, 1]` | Distributed Data Parallel across both GPUs |
| `degrees` | 2.0 | Small rotation to simulate natural handwriting tilt |
| `shear` | 2.0 | Mild shear augmentation for script-style generalisation |
| `mosaic` | 1.0 | Full mosaic; beneficial for dense multi-character scenes |

The best model checkpoint is saved to `/kaggle/working/runs/detect/`.

In [6]:
import os
import shutil
import random
import yaml
from ultralytics import YOLO

# --- 6. LAUNCH YOLO STAGE 1 TRAINING ---
# (Ensure you selected GPU T4 x2 or P100 in your Kaggle Notebook settings!)
model = YOLO('yolov8s.pt') # Start with YOLOv8 Small

results = model.train(
    data=yaml_path,
    epochs=10,               
    imgsz=1024,              # Target A4 resolution scaling
    batch=8,                 # Depending on Kaggle GPU, you may need to lower this to 4 if OOM occurs
    device=[0,1],                
    
    # YOLO's Built-in Augmentations
    degrees=2.0,             
    shear=2.0,               
    perspective=0.001,       
    mosaic=1.0,              
    erasing=0.2              
)

Ultralytics 8.4.58 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/Custom_Dataset/data.yaml, degrees=2.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.2, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-6, 

## Step 7 — Clear VRAM Between Stages

After Stage 1 finishes, free GPU memory before loading Stage 2. DDP spawns a child process that holds references; `gc.collect()` and `torch.cuda.empty_cache()` ensure those references are released before Stage 2 initialises.

In [14]:
import torch
import gc

# 1. Force Python to delete unreferenced variables in RAM
gc.collect()

# 2. Tell PyTorch to empty the VRAM cache on all GPUs
torch.cuda.empty_cache()

# 3. Specifically target and clear GPU 1 (the second GPU)
with torch.cuda.device(1):
    torch.cuda.empty_cache()
    
# 4. Specifically target and clear GPU 0 (the first GPU)
with torch.cuda.device(0):
    torch.cuda.empty_cache()

# Print current memory usage to verify it worked
print(f"GPU 0 Memory Allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
if torch.cuda.device_count() > 1:
    print(f"GPU 1 Memory Allocated: {torch.cuda.memory_allocated(1) / 1024**3:.2f} GB")

GPU 0 Memory Allocated: 0.18 GB
GPU 1 Memory Allocated: 0.00 GB


## Step 8 — Stage 2 Configuration: Fine-tuning on Real Data

The competition training set contains a small number of real annotated handwriting images (17 images, 6,679 character annotations). This stage fine-tunes the Stage 1 checkpoint on this real data, allowing the detector to adapt from synthetic rendering statistics to authentic handwriting variability.

> **Dataset path:** `REAL_DATA_DIR` points to the competition's annotated image set. Adjust this path if the dataset is mounted at a different location.

In [4]:
# --- COMPETITION DATASET PATHS ---
# These paths point to the Comsys 7 competition dataset mounted on Kaggle.
# To access them: on your Kaggle notebook page, click '+ Add Data' and
# search for the competition dataset 'handwritten-multiscript-character-segmentation-recognition'.
# If running locally, update these paths to where you have extracted the competition data.

import os
import glob
import json
import shutil
import random
from PIL import Image
from ultralytics import YOLO

# --- 1. CONFIGURATION ---
JSON_DIR = "/kaggle/input/competitions/handwritten-multiscript-character-segmentation-recognition/JU_CMATER/train/annotations"
IMG_DIR = "/kaggle/input/competitions/handwritten-multiscript-character-segmentation-recognition/JU_CMATER/train/images"
FINETUNE_DIR = "/kaggle/working/finetune_dataset"

### Step 8a — Build Fine-tune Directory Structure

Create the YOLO-compatible folder layout for the real-data fine-tuning stage. Annotation JSON files (LabelMe format) are converted to normalised YOLO bounding-box `.txt` files with all coordinates clamped to `[0, 1]` to prevent out-of-bounds errors. A 14/3 train–validation split is applied given the limited image count.

In [6]:
for folder in ['images/train', 'images/val', 'labels/train', 'labels/val']:
    os.makedirs(os.path.join(FINETUNE_DIR, folder), exist_ok=True)

# --- 2. PARSE AND CONVERT ANNOTATIONS ---
print("Extracting JSON annotations into Class-Agnostic YOLO format...")
json_files = glob.glob(os.path.join(JSON_DIR, "*.json"))
random.shuffle(json_files)

# With only 17 images, a 14 Train / 3 Val split is optimal
train_cutoff = 14  
processed_count = 0

for i, json_path in enumerate(json_files):
    base_name = os.path.splitext(os.path.basename(json_path))[0]
    
    # Locate matching image (handling variable extensions)
    img_path = None
    for ext in ['.jpg', '.JPG', '.png', '.jpeg']:
        temp = os.path.join(IMG_DIR, f"{base_name}{ext}")
        if os.path.exists(temp):
            img_path = temp
            break
            
    if not img_path: 
        print(f"Missing image for {base_name}. Skipping.")
        continue
        
    img = Image.open(img_path)
    img_w, img_h = img.width, img.height
    
    split = 'train' if i < train_cutoff else 'val'
    
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    yolo_lines = []
    
    for shape in data.get('shapes', []):
        points = shape.get('points')
        if not points or len(points) < 2: 
            continue
            
        # Extract Min/Max bounds from the polygon points
        xs = [p[0] for p in points]
        ys = [p[1] for p in points]
        xmin, xmax = min(xs), max(xs)
        ymin, ymax = min(ys), max(ys)
        
        # YOLO Normalization Math
        xc = ((xmin + xmax) / 2.0) / img_w
        yc = ((ymin + ymax) / 2.0) / img_h
        w = (xmax - xmin) / img_w
        h = (ymax - ymin) / img_h
        
        # Clamp values between 0.0 and 1.0 to prevent out-of-bounds errors
        xc, yc = max(0, min(1, xc)), max(0, min(1, yc))
        w, h = max(0, min(1, w)), max(0, min(1, h))
        
        # FORCE CLASS TO 0
        yolo_lines.append(f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")
        
    if yolo_lines:
        shutil.copy(img_path, os.path.join(FINETUNE_DIR, 'images', split, os.path.basename(img_path)))
        with open(os.path.join(FINETUNE_DIR, 'labels', split, f"{base_name}.txt"), 'w') as f:
            f.writelines(yolo_lines)
        processed_count += 1

Extracting JSON annotations into Class-Agnostic YOLO format...


### Step 8b — Generate Stage 2 Dataset YAML

Write the dataset configuration YAML for the fine-tuning stage. The class list remains a single class (`Character`) consistent with Stage 1.

In [7]:
# --- 3. GENERATE YAML ---
yaml_path = os.path.join(FINETUNE_DIR, 'dataset.yaml')
with open(yaml_path, 'w') as f:
    f.write(f"path: {FINETUNE_DIR}\ntrain: images/train\nval: images/val\nnc: 1\nnames: ['Character']\n")

print(f"Successfully processed {processed_count} real images into {FINETUNE_DIR}")

# --- 4. THE STAGE 2 FINE-TUNING EXECUTION ---
print("Initializing Stage 2 Fine-Tuning...")

# LOAD THE BRAIN YOU JUST FINISHED TRAINING
# --- PATH TO ADJUST ---
# After Stage 1 training, YOLO saves the run to /kaggle/working/runs/detect/<run_name>/
# The run name (e.g. "train-6") is printed at the end of Stage 1 training.
# Update the path below to match your actual run folder name.
pretrained_weights = "/kaggle/working/runs/detect/train-6/weights/best.pt"  # <-- UPDATE run name if different

if not os.path.exists(pretrained_weights):
    print(f"[ERROR] Could not find Phase 1 weights at {pretrained_weights}")
else:
    model = YOLO(pretrained_weights)

    results = model.train(
        data=yaml_path,
        epochs=100,               
        imgsz=1024,
        batch=4,                  
        
        # --- THE FIX: Force the Optimizer ---
        device=0,                 # CRITICAL: Use only 1 GPU for tiny datasets
        amp=False,
        optimizer='AdamW',        # Explicitly setting this disables the 'auto' override!
        lr0=0.0001,               # YOLO will now strictly obey this micro-learning rate
        lrf=0.01,                 
        warmup_epochs=0,          
        
        # Gentle Augmentations 
        degrees=1.0,             
        shear=1.0,               
        mosaic=0.5,              
        erasing=0.1,
        
        project='/kaggle/working/runs',
        name='finetune_real_17_stable'
    )
    
    print("Fine-tuning complete! Your final text-detection eye is ready.")

Successfully processed 17 real images into /kaggle/working/finetune_dataset
Initializing Stage 2 Fine-Tuning...
Ultralytics 8.4.58 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/finetune_dataset/dataset.yaml, degrees=1.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.1, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/runs/detect/train-6

### Final Annotations Output

The final bounding box annotations from this YOLO detection pipeline are saved to `intermediate_boxes_expanded.json`.
This JSON file will be provided as an input to the next stage of the pipeline: `text-classification.ipynb`.

## Step 9 — Inference and Bounding Box Export

Run the trained YOLO detector over all competition test images to extract candidate character bounding boxes. Key inference settings:

| Setting | Value | Rationale |
| :--- | :--- | :--- |
| `conf` | 0.08 | Low threshold maximises recall — missed characters are more costly than false positives |
| `iou` | 0.45 | Standard NMS overlap threshold |
| `max_det` | 2000 | High limit to capture all characters on dense handwriting pages |

Detected boxes are expanded by a small padding factor and saved to **`intermediate_boxes_expanded.json`**. This file is the primary input to the next stage: `text-classification.ipynb`.

In [2]:
import os
import glob
import json
from ultralytics import YOLO

# --- CONFIGURATION ---
TEST_IMG_DIR = '/kaggle/input/competitions/handwritten-multiscript-character-segmentation-recognition/JU_CMATER/test/images'
YOLO_WEIGHTS = '/kaggle/working/runs/finetune_real_17_stable-2/weights/best.pt' # Update this if your path is different!

print("👁️ Loading YOLO Stage 1 Model...")
yolo_model = YOLO(YOLO_WEIGHTS)

test_images = glob.glob(os.path.join(TEST_IMG_DIR, '*.*'))
all_page_boxes = {}

print(f"Extracting bounding boxes from {len(test_images)} test images...")

for img_path in test_images:
    base_name = os.path.splitext(os.path.basename(img_path))[0]
    
    # Very low confidence to maximize recall!
    results = yolo_model(img_path, conf=0.08, iou=0.45,max_det = 2000, verbose=False)
    
    # Extract the box coordinates [x1, y1, x2, y2] and convert to a standard Python list
    boxes = results[0].boxes.xyxy.cpu().numpy().tolist()
    
    all_page_boxes[base_name] = boxes
    print(f" - Page {base_name}: Found {len(boxes)} potential characters.")

# Save to an intermediate file
output_path = '/kaggle/working/intermediate_boxes_expanded.json'
with open(output_path, 'w') as f:
    json.dump(all_page_boxes, f)

print(f"\n✅ Extraction Complete! Download '{output_path}' and upload it to Notebook 2.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
👁️ Loading YOLO Stage 1 Model...
Extracting bounding boxes from 6 test images...
 - Page 102: Found 231 potential characters.
 - Page 101: Found 264 potential characters.
 - Page 010: Found 1003 potential characters.
 - Page 001: Found 865 potential characters.
 - Page 005: Found 757 potential characters.
 - Page 019: Found 1455 potential characters.

✅ Extraction Complete! Download '/kaggle/working/intermediate_boxes_expanded.json' and upload it to Notebook 2.
